# Research Backtest — Generic

Backtest L/S pour n'importe quelle feature enregistrée dans `features.yaml`.

**Usage :** change `FEATURE_ID` en section 0, exécute toutes les cellules.

Sections :
1. Chargement données + calcul feature
2. Signal binaire L/S par asset
3. Portefeuille cross-sectionnel proportionnel
4. Métriques et visualisations

In [ ]:
import pathlib, sys, importlib
ROOT = pathlib.Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yaml
import warnings
warnings.filterwarnings('ignore')

# ── Paramètre principal ───────────────────────────────────────────────────────
FEATURE_ID = 'reversal_5d'   # ← CHANGE ICI

# ── Config ───────────────────────────────────────────────────────────────────
RAW      = ROOT / 'data' / 'raw'
EXCHANGE = 'binance-futures'
ASSETS   = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'LTCUSDT', 'BNBUSDT']

plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False,
})

# ── Charge la config de la feature depuis features.yaml ──────────────────────
with open(ROOT / 'configs' / 'features.yaml') as f:
    registry = yaml.safe_load(f)['features']

feat_cfg = next((f for f in registry if f['id'] == FEATURE_ID), None)
assert feat_cfg is not None, f'Feature {FEATURE_ID} introuvable dans features.yaml'

params = feat_cfg.get('params', {})
print(f'Feature  : {FEATURE_ID}')
print(f'Famille  : {feat_cfg["family"]}')
print(f'Source   : {feat_cfg["source"]}')
print(f'Params   : {params}')

## 1. Chargement des données & calcul du signal

In [ ]:
def load_ohlcv(symbol: str) -> pd.DataFrame:
    base  = RAW / f'exchange={EXCHANGE}' / 'data_type=ohlcv_1d' / f'symbol={symbol}'
    parts = sorted(base.rglob('part-*.parquet'))
    if not parts:
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df['date'] = pd.to_datetime(df['ts_open'], unit='us', utc=True).dt.normalize()
    return df.sort_values('date').drop_duplicates('date').set_index('date')

mod = importlib.import_module(f'dfi_features.{FEATURE_ID}')

data = {}
for sym in ASSETS:
    raw = load_ohlcv(sym)
    if raw.empty:
        print(f'  {sym}: aucune donnée')
        continue
    df             = pd.DataFrame(index=raw.index)
    df['close']    = raw['close'].astype(float)
    df['ret_1d']   = df['close'] / df['close'].shift(1) - 1
    df['signal_raw'] = mod.compute(raw, params)   # valeur brute de la feature
    df['ret_fwd']  = df['ret_1d'].shift(-1)
    data[sym] = df
    n_valid = df['signal_raw'].notna().sum()
    print(f'  {sym}: {len(df)} jours  |  signal valide: {n_valid} barres  '
          f'({df.index[0].date()} → {df.index[-1].date()})')

### Aperçu du signal

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(15, 4), sharey=True)
for ax, (sym, df) in zip(axes, data.items()):
    s = df['signal_raw'].dropna()
    ax.hist(s, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='k', lw=1)
    ax.set_title(f'{sym}\nμ={s.mean():.3f}  σ={s.std():.3f}', fontsize=8)
    ax.set_xlabel(FEATURE_ID)
axes[0].set_ylabel('Fréquence')
fig.suptitle(f'Distribution du signal — {FEATURE_ID}', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Backtest L/S binaire par asset

- `signal = +1` si feature > 0, `-1` sinon
- `strat_ret = signal × ret_fwd`

In [ ]:
def sharpe(r, freq=252):
    r = r.dropna()
    return (r.mean() / r.std()) * np.sqrt(freq) if r.std() > 0 else float('nan')

def max_dd(pv):
    rm = pv.cummax()
    return ((pv - rm) / rm).min()

def win_rate(r):
    r = r.dropna()
    return (r > 0).sum() / len(r)

for sym, df in data.items():
    df['signal']     = np.where(df['signal_raw'] > 0, 1.0, -1.0)
    df['strat_ret']  = df['signal'] * df['ret_fwd']
    df['portf_value'] = (1 + df['strat_ret'].fillna(0)).cumprod()
    df['bh_value']    = (1 + df['ret_1d'].fillna(0)).cumprod()

rows = []
for sym, df in data.items():
    sr = df['strat_ret'].dropna()
    rows.append({
        'Asset':        sym,
        'Sharpe strat': round(sharpe(sr), 3),
        'Sharpe B&H':   round(sharpe(df['ret_1d'].dropna()), 3),
        'Return (%)':   round((df['portf_value'].iloc[-1] - 1) * 100, 1),
        'Max DD (%)':   round(max_dd(df['portf_value']) * 100, 1),
        'Win rate (%)': round(win_rate(sr) * 100, 1),
        'N jours':      len(sr),
    })

summary = pd.DataFrame(rows).set_index('Asset')
print(f'Backtest L/S binaire — {FEATURE_ID}\n')
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(13, 3 * len(data)), sharex=True)
for ax, (sym, df) in zip(axes, data.items()):
    v = df[['portf_value', 'bh_value']].dropna()
    ax.plot(v.index, v['portf_value'], lw=1.4, color='steelblue', label='Stratégie')
    ax.plot(v.index, v['bh_value'],    lw=1.0, color='gray', alpha=0.5, label='B&H')
    ax.axhline(1, color='k', lw=0.6, ls='--')
    sh = sharpe(df['strat_ret'].dropna())
    ax.set_title(f'{sym}  Sharpe={sh:.2f}  Return={(df["portf_value"].iloc[-1]-1)*100:.1f}%', fontsize=9)
    ax.set_ylabel('Valeur')
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
fig.suptitle(f'L/S binaire — {FEATURE_ID}', fontsize=11)
plt.gcf().autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

## 3. Portefeuille cross-sectionnel proportionnel

```
w_i(t) = signal_raw_i(t) / Σ |signal_raw_j(t)|
port_ret(t) = Σ w_i(t) × ret_fwd_i(t)
```

Dollar-neutral : Σ w_i = 0. Les assets avec le signal le plus fort reçoivent la plus grosse position.

In [ ]:
common_idx = pd.DatetimeIndex(sorted(set.union(*[set(df.index) for df in data.values()])))

sig_panel = pd.DataFrame({sym: data[sym]['signal_raw'].reindex(common_idx) for sym in ASSETS})
ret_panel = pd.DataFrame({sym: data[sym]['ret_fwd'].reindex(common_idx)    for sym in ASSETS})

abs_sum  = sig_panel.abs().sum(axis=1).replace(0, np.nan)
weights  = sig_panel.div(abs_sum, axis=0)

port_ret   = (weights * ret_panel).sum(axis=1)
port_value = (1 + port_ret.fillna(0)).cumprod()

# Binaire moyen pour comparaison
bin_rets  = pd.DataFrame({sym: data[sym]['strat_ret'].reindex(common_idx) for sym in ASSETS})
bin_mean  = bin_rets.mean(axis=1)
bin_value = (1 + bin_mean.fillna(0)).cumprod()

sh_prop = sharpe(port_ret)
sh_bin  = sharpe(bin_mean)

sep = '-' * 50
print(sep)
print(f'  Proportionnel  Sharpe = {sh_prop:+.3f}  Return = {(port_value.iloc[-1]-1)*100:.1f}%')
print(f'  Binaire (moy.) Sharpe = {sh_bin:+.3f}  Return = {(bin_value.iloc[-1]-1)*100:.1f}%')
print(f'  Max DD proportionnel  = {max_dd(port_value)*100:.1f}%')
print(sep)

# Poids courants
w_now = weights.dropna().tail(1).round(3)
w_now.index = w_now.index.strftime('%Y-%m-%d')
print("\nPoids aujourd'hui :")
print(w_now.to_string())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
colors_assets = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

# Panel 1 : P&L cumulé
ax = axes[0]
ax.plot(port_value.index, port_value.values, lw=1.6, color='steelblue',
        label=f'Proportionnel  Sharpe={sh_prop:.2f}')
ax.plot(bin_value.index,  bin_value.values,  lw=1.0, color='darkorange', alpha=0.7,
        label=f'Binaire (moy.) Sharpe={sh_bin:.2f}')
ax.axhline(1, color='k', lw=0.6, ls='--')
ax.set_ylabel('Valeur (base 1)')
ax.set_title('Proportionnel vs Binaire — valeur cumulée')
ax.legend(fontsize=8)

# Panel 2 : poids dans le temps
ax = axes[1]
for sym, col in zip(ASSETS, colors_assets):
    w = weights[sym].dropna()
    ax.plot(w.index, w.values, lw=0.8, color=col, alpha=0.7, label=sym)
ax.axhline(0, color='k', lw=0.6)
ax.set_ylabel('Poids w_i(t)')
ax.set_title('Évolution des poids par asset')
ax.legend(fontsize=7, ncol=5)

# Panel 3 : rolling Sharpe 63j
ax = axes[2]
roll_sh = port_ret.rolling(63, min_periods=20).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(252) if x.std() > 0 else np.nan
)
ax.plot(roll_sh.index, roll_sh.values, lw=1.2, color='steelblue')
ax.fill_between(roll_sh.index, 0, roll_sh.values,
                where=roll_sh.values >= 0, color='tab:green', alpha=0.15)
ax.fill_between(roll_sh.index, 0, roll_sh.values,
                where=roll_sh.values < 0,  color='tab:red',   alpha=0.15)
ax.axhline(0, color='k', lw=0.6)
ax.axhline(1, color='tab:green', lw=0.8, ls='--', alpha=0.5, label='Sharpe=1')
ax.set_ylabel('Rolling Sharpe (63j)')
ax.set_title('Stabilité — Rolling Sharpe 63j')
ax.legend(fontsize=7)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))

fig.suptitle(f'Portefeuille proportionnel — {FEATURE_ID}', fontsize=11)
plt.gcf().autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()